## Tree Based Models

In [1]:
import pandas as pd
import site
import sys

sys.path.append(site.getusersitepackages())

In [2]:
df = pd.read_csv("../Data/Cleaned/customer_churn_dataset.csv")
df.head(5)

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1.0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
1,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
2,2.0,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
3,3.0,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
4,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0


## Feature Engineering

In [3]:
import numpy as np

df["Issue_Level"] = np.where(
    df["Support Calls"] <= 2,
    "Low Issues",
    np.where(df["Support Calls"] <= 4,
             "Medium Issues",
             "High Issues")
)

df["Delay_Level"] = np.where(
    df["Payment Delay"] <= 15,
    "Low Delay",
    np.where(df["Payment Delay"] <= 20,
             "Medium Delay",
             "High Delay")
)

df["Spend_Level"] = np.where(
    df["Total Spend"] <= 508,
    "Low Spend",
    "High Spend"
)

In [4]:
X = df[["Issue_Level","Delay_Level","Spend_Level","Contract Length"]]
y = df["Churn"]
x = pd.get_dummies(
    X,
    drop_first=False,
    dtype= int
)

In [5]:
from sklearn.model_selection import train_test_split
x_train,x_test,X_train_original,X_test_original,y_train,y_test = train_test_split(x,X,y,test_size=0.2,random_state=42,stratify = y)

## Decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model1 = DecisionTreeClassifier()


### Hyperparamter tuning using grid search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"min_samples_split":[2,5,20,35,50],
              "criterion":["gini","entropy"],
              "splitter":["best","random"],
              "max_depth":[3, 5, 7, 10, 15, None],
              "min_samples_leaf":[1, 2, 5, 10, 20],
               "max_features": [None, "sqrt", "log2"]}

grid = GridSearchCV(
    estimator=model1,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)
grid.fit(x_train,y_train)

best_model = grid.best_estimator_


In [ ]:
print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

Best Parameters: {'criterion': 'gini', 'max_depth': 5, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'splitter': 'best'}
Best CV Accuracy: 0.8973881871567627


#### Observations and conclusion

1. The criteria gini outperformed entropy
2. max_depth is the level to which a tree can grow. if it kept small then tree underfit or if kept high may overfit.
    - The grid search found max_depth 4 where it is neither overfitting nor underfitting.
3. min_sample_split is the minimum number of samples should a node have to be able to split. low value of it may even try to split with the sample of just 2 customers which might be just randomly occured leading to overfitting. again high value may lead to  high bias making model simple not catching the true pattern.
    - The grid search found min_sample_split 2. Potentially may lead overfitting.
4. min_sample_leaf is the minimum number samples should a leaf node have after splitting.
    - The grid search found min_samples_leaf 1. Potentially may lead overfitting.
5. The best CV accuracy is 0.897

In [ ]:
from joblib import dump

dump(best_model,  "decision_tree.joblib")

['decision_tree.joblib']

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Prediction
y_pred = best_model.predict(x_test)

# Probability Prediction
y_prob = best_model.predict_proba(x_test)[:,1]

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8967360107677995
Precision: 0.8866703923859846
Recall   : 0.9332964936986399
F1 Score : 0.9093861812623754
ROC AUC  : 0.9231559265609711

Confusion Matrix
[[38251  6692]
 [ 3742 52357]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.91      0.85      0.88     44943
         1.0       0.89      0.93      0.91     56099

    accuracy                           0.90    101042
   macro avg       0.90      0.89      0.89    101042
weighted avg       0.90      0.90      0.90    101042



## Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### Hyperparamter tuning using Random Search 

In [ ]:
param_grid1 = {
    "n_estimators": [100, 200, 300],
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid1,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

In [ ]:
random_search.fit(x_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'criterion': ['gini', 'entropy'], 'max_depth': [5, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used he

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Prediction
y_pred = best_model.predict(x_test)

# Probability Prediction
y_prob = best_model.predict_proba(x_test)[:,1]

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8967360107677995
Precision: 0.8866703923859846
Recall   : 0.9332964936986399
F1 Score : 0.9093861812623754
ROC AUC  : 0.9233613036513193

Confusion Matrix
[[38251  6692]
 [ 3742 52357]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.91      0.85      0.88     44943
         1.0       0.89      0.93      0.91     56099

    accuracy                           0.90    101042
   macro avg       0.90      0.89      0.89    101042
weighted avg       0.90      0.90      0.90    101042



In [ ]:
from joblib import dump

dump(best_model, "random_forest_model.joblib")

['random_forest_model.joblib']

In [ ]:
print(random_search.best_params_)

{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None, 'criterion': 'entropy'}


#### Observations and conclusion
1. The n_estimators is the number of trees used by random forest.
    -  The random search found 200 trees perform best.
2. The min_sample_split and min_samples_leaf are 2 and 4 greater than decision tree avoiding the risk of overfitting.

## Xgboost

In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [7]:
model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

In [8]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [9]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

In [11]:
random_search.fit(x_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also

In [ ]:
best_model = random_search.best_estimator_

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

Best Parameters: {'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
Best CV Accuracy: 0.8973881871567627


#### Observations and Conclusion

1. **n_estimators** is the number of boosting rounds or trees used by XGBoost.
   - The random search found 100 trees perform best.

2. **max_depth** is the maximum depth of each tree. If kept small then tree underfits or if kept high may overfit.
   - The random search found max_depth 5 where it is neither overfitting nor underfitting, similar to decision tree but with better generalization due to boosting.

3. **learning_rate** (eta) controls the contribution of each tree to the final model. Lower values make the model more robust to overfitting but require more trees, while higher values speed up learning but risk overfitting.
   - The random search found learning_rate 0.2 as optimal, balancing convergence speed and generalization.

4. **subsample** is the fraction of training data randomly sampled for each tree. Lower values prevent overfitting by introducing randomness, while higher values use more data but may overfit.
   - The random search found subsample 0.8, meaning 80% of training data is used per tree, reducing overfitting risk while maintaining sufficient data for learning.

5. **colsample_bytree** is the fraction of features randomly sampled for each tree. Lower values increase diversity among trees and reduce overfitting, similar to Random Forest's feature sampling.
   - The random search found colsample_bytree 0.8, using 80% of features per tree, which helps in reducing overfitting and improving generalization.

6. **min_child_weight** is the minimum sum of instance weights (hessian) required in a child node. Higher values prevent overfitting by making splits more conservative, while lower values allow capturing more complex patterns.
   - The random search found min_child_weight 3, which is greater than the default 1, successfully avoiding the risk of overfitting by ensuring splits are based on sufficient data.

7. The best CV accuracy is 0.8973881871567627, which is comparable to the decision tree (0.897) but with better generalization potential due to XGBoost's regularization and ensemble nature.

---

**Note**: The XGBoost model achieves similar accuracy to the decision tree (both ~89.7%) but with superior generalization capabilities due to:
- **Regularization** (through min_child_weight and subsampling)
- **Boosting ensemble** (sequential error correction)
- **Feature subsampling** (colsample_bytree for diversity)

This suggests the dataset may have reached a performance ceiling around 89.7%, and XGBoost provides a more robust solution with less overfitting risk compared to the single decision tree.

In [ ]:
y_pred = best_model.predict(x_test)
y_prob = best_model.predict_proba(x_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy : 0.8967360107677995
Precision: 0.8866703923859846
Recall   : 0.9332964936986399
F1 Score : 0.9093861812623754
ROC AUC  : 0.9234000914409626

Confusion Matrix
[[38251  6692]
 [ 3742 52357]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.91      0.85      0.88     44943
         1.0       0.89      0.93      0.91     56099

    accuracy                           0.90    101042
   macro avg       0.90      0.89      0.89    101042
weighted avg       0.90      0.90      0.90    101042



## Evaluation of Perfomace metric 
## Confusion Matrix

|                | Predicted: No | Predicted: Yes |
|----------------|---------------|----------------|
| **Actual: No** | 38,251        | 6,692          |
| **Actual: Yes**| 3,742         | 52,357         |

---

### Interpretation

| Metric | Value |
|--------|-------|
| **True Negatives (TN)** | 38,251 |
| **False Positives (FP)** | 6,692 |
| **False Negatives (FN)** | 3,742 |
| **True Positives (TP)** | 52,357 |


#### Observations and conclusion

1. The all tree based models have acheived the same confusion matrix but random forest and Xgboost provide better generalisation and dataset may have reached the maximum performance limit. These models are capturing the most dominant patterns.
2. The tree based models miss 3,742 churn customers which can be fatal if they high value profile customers. Though it is much lower than logistic regression allowing the model to capture more non-linear relationships.
3. The models missclassify 6,692 non-churning customers which may lead to retetion stratergies to wrong segments causing financial loss as we discussed in logistic regression section.
4. These models achieve the accuracy of 0.89 with precision of 0.886.
5. The model achieves the recall of 0.933 suggesting more capable of capuring churn customers.  



## Segmentwise Metric Evaluation

In [ ]:
results = X_test_original.copy()

results["Actual"] = y_test.values
results["Predicted"] = y_pred

In [ ]:
false_negative = results[
    (results["Actual"] == 1) &
    (results["Predicted"] == 0)
]

false_negative.head()

,Issue_Level,Delay_Level,Spend_Level,Contract Length,Actual,Predicted
130263,Medium Issues,Medium Delay,High Spend,Quarterly,1.0,0
194081,Medium Issues,Low Delay,High Spend,Quarterly,1.0,0
213143,Medium Issues,Medium Delay,High Spend,Quarterly,1.0,0
252515,Medium Issues,Low Delay,High Spend,Annual,1.0,0
92707,Medium Issues,Low Delay,High Spend,Annual,1.0,0


In [ ]:
false_positive = results[
    (results["Actual"] == 0) &
    (results["Predicted"] == 1)
]

false_positive.head()

,Issue_Level,Delay_Level,Spend_Level,Contract Length,Actual,Predicted
5192,Medium Issues,Low Delay,High Spend,Monthly,0.0,1
50894,Medium Issues,Low Delay,Low Spend,Quarterly,0.0,1
90680,Low Issues,High Delay,High Spend,Annual,0.0,1
18573,Medium Issues,Low Delay,Low Spend,Monthly,0.0,1
82807,Medium Issues,Low Delay,Low Spend,Quarterly,0.0,1


In [ ]:
fp_rate = (
    false_positive["Spend_Level"].value_counts()
    /
    X_test_original[y_test == 0]["Spend_Level"].value_counts()
)

print((fp_rate * 100).round(2))

Spend_Level
High Spend      7.61
Low Spend     100.00
Name: count, dtype: float64


In [ ]:
fn_rate = (
    false_negative["Spend_Level"].value_counts()
    /
    X_test_original[y_test == 1]["Spend_Level"].value_counts()
)

print((fn_rate * 100).round(2))

Spend_Level
High Spend    12.75
Low Spend       NaN
Name: count, dtype: float64


In [ ]:
fp_rate = (
    false_positive["Issue_Level"].value_counts()
    /
    X_test_original[y_test == 0]["Issue_Level"].value_counts()
)

print((fp_rate * 100).round(2))

Issue_Level
High Issues      100.00
Low Issues         6.81
Medium Issues     10.90
Name: count, dtype: float64


In [ ]:
fp_rate = (
    false_negative["Issue_Level"].value_counts()
    /
    X_test_original[y_test == 1]["Issue_Level"].value_counts()
)

print((fp_rate * 100).round(2))

Issue_Level
High Issues        NaN
Low Issues       14.18
Medium Issues    19.20
Name: count, dtype: float64


In [ ]:
fp_rate = (
    false_positive["Delay_Level"].value_counts()
    /
    X_test_original[y_test == 0]["Delay_Level"].value_counts()
)

print((fp_rate * 100).round(2))

Delay_Level
High Delay      100.00
Low Delay        13.31
Medium Delay      9.03
Name: count, dtype: float64


In [ ]:
fp_rate = (
    false_negative["Delay_Level"].value_counts()
    /
    X_test_original[y_test == 1]["Delay_Level"].value_counts()
)

print((fp_rate * 100).round(2))

Delay_Level
High Delay        NaN
Low Delay       11.03
Medium Delay     9.76
Name: count, dtype: float64


In [ ]:
fp_rate = (
    false_negative["Contract Length"].value_counts()
    /
    X_test_original[y_test == 1]["Contract Length"].value_counts()
)

print((fp_rate * 100).round(2))

Contract Length
Annual       10.54
Monthly        NaN
Quarterly     9.98
Name: count, dtype: float64


In [ ]:
fp_rate = (
    false_positive["Contract Length"].value_counts()
    /
    X_test_original[y_test == 0]["Contract Length"].value_counts()
)

print((fp_rate * 100).round(2))

Contract Length
Annual        10.67
Quarterly     10.54
Monthly      100.00
Name: count, dtype: float64


#### Observations for False Negatives

1. The  tree based models misses 10.54%, 0% and 9.98% Annual, Monthly and Quarterly Churning Customers. It captures all monthly contract length churning customers.
2. The model misses 0%, 11.03% and 9.76% High, Low and Medium delay Churning Customers.
3. The model misses 0%, 14.18% and 19.2% High, Low and Medium Issues Churnning Customers.
4. The model misses 12.75% and 0% High and Low Spending Churnning Customers.

#### Conclusion

- The model's false negatives are primarily concentrated among high-value customers with Quarterly/Annual contracts, low to medium payment delays, and low to medium support issues. Since these customers appear relatively healthy despite eventually churning, the model fails to identify them as at risk. Losing these customers can be particularly costly because they contribute higher revenue and may have greater customer lifetime value.

#### Observations for False positives

1. The model misses 10.67%, 100% and 10.54% Annual, Monthly and Quarterly Non-churning Customers.
2. The model misses 100%, 13.31% and 9.03% High, Low and Medium delay Non-churning Customers.
3. The model misses 100%, 6.81% and 10.09% High, Low and Medium Issues Non-churning Customers.
4. The model misses 7.61% and 100% High and Low Spending Non-churning customers.

#### Conclusion

- Applying the retention stratergies on missed non-churning low spend,monthly contract length low value profile customers can lead to a buisness loss.

- The model appears to rely heavily on strong churn indicators such as high payment delays or frequent support issues. Customers who churned without exhibiting these dominant patterns were more likely to be misclassified as non-churn.